# Veri Hazırlama

**Proje:** ABD Trafik Kazalarında Hava Durumu, Yol Tipi ve Saat Bilgisine Göre
Kaza Şiddetinin Tahmini

**Kapsam:** Florida, New York, Minnesota (farklı iklimleri temsil ediyorlar).

**Girdi:** `US_Accidents_March23.csv`, `county_referans_kendi.csv`
**Çıktı:** `temiz_veri.parquet`

| Bölüm | Yapılan iş |
|---|---|
| 1 | Ham veriyi oku, üç eyaleti seç, kullanılmayan sütunları at |
| 2 | Tekrar eden kayıtları temizle |
| 3 | Hatalı ölçümleri işaretle, eksikleri doldur |
| 4 | İlçe nüfus / kentsel-kırsal verisini birleştir |
| 5 | Zaman, yol tipi, hava durumu değişkenlerini türet |
| 6 | Kalite denetimi ve kaydetme |

> **Veri hazırlığı yalnızca bu notebook'ta yapılır.** 02 ve 03 dosyayı okuyup
> doğrudan kullanır, hiçbir temizlik işlemini tekrarlamaz. Böylece EDA'nın
> çalıştığı veri ile modelin gördüğü veri ayrışmaz.

## 1. Ham verinin okunması

Dosya 2.9 GB; 500.000 satırlık parçalar hâlinde okunup üç eyaletin kayıtları
alınıyor.

Atılan sütunlardan `Distance(mi)` ve `End_Time` **hedef sızıntısı** taşır:
ikisi de kaza olduktan sonra ölçülür, yani kaza anında bilinemez.

### 1.1 Zaman damgası

`Start_Time` veride üç ayrı metin biçiminde yazılı (`...15:36:03`,
`...:00.000000`, `...:16.000000000`); 66.290 zaman anı birden fazla biçimde
kayıtlı. Metin farklı, zaman aynı.

Sütun bu yüzden **her karşılaştırmadan önce** tarih tipine çevriliyor. Sonraya
bırakılırsa § 2'deki tekrar analizi aynı kazanın farklı yazılmış iki kaydını
ayrı olay sayar.

In [31]:
#Uzun süren işlemler öncesinde girdi dosyalarının varlığını doğrulama
import pandas as pd
import numpy as np
import re
import os

RAW  = "../data/raw/"
PROC = "../data/processed/"

ANA_VERI = RAW + "US_Accidents_March23.csv"
REFERANS = RAW + "county_referans_kendi.csv"

EYALETLER = ["FL", "NY", "MN"]

for yol in [ANA_VERI, REFERANS]:#yol bir geçici değişken. Her turda listeden bir sonraki öğeyi alıyor:
    if os.path.exists(yol):#diske bakar, dosya varsa True döner.
        mb = os.path.getsize(yol) / 1024**2#	Boyut (bayt)
        print(f"✓ {os.path.basename(yol):<32} {mb:>8.1f} MB")
    else:
        print(f"✗ BULUNAMADI: {yol}")

os.makedirs(PROC, exist_ok=True)#processed klasörünü oluşturuyor.

✓ US_Accidents_March23.csv           2916.5 MB
✓ county_referans_kendi.csv             0.2 MB


In [32]:
ATILACAK = ["End_Lat", "End_Lng", "Wind_Chill(F)", "Distance(mi)",
            "End_Time", "Description", "Country", "Source",
            "Turning_Loop", "Weather_Timestamp", "Airport_Code"]

parcalar = []
okuyucu = pd.read_csv(ANA_VERI, chunksize=500_000, low_memory=False)

for i, parca in enumerate(okuyucu, 1):
    secili = parca[parca["State"].isin(EYALETLER)]
    parcalar.append(secili)
    print(f"{i:>2}. parça — {len(secili):>7,} satır alındı")

df = pd.concat(parcalar, ignore_index=True)
df = df.drop(columns=ATILACAK)

del parcalar #Döngü bitti ama parcalar listesi hâlâ bellekte,del onu siliyor, yaklaşık 1 GB boşalıyor.

print()
print("Toplam satır:", f"{len(df):,}")
print("Sütun sayısı:", df.shape[1])
print()
print(df["State"].value_counts())

# --- ZAMAN DAMGASINI HEMEN STANDARTLAŞTIR (bkz. § 1.1) ---
# Aynı an veride üç ayrı metin biçiminde yazılı olabiliyor. Metin olarak
# farklı, zaman olarak aynı. Dönüşüm burada yapılmazsa § 2'deki tekrar
# karşılaştırması aynı kazanın iki kaydını farklı sanar.
print("\n--- Start_Time metin uzunlukları (dönüşümden önce) ---")
uzunluk = df["Start_Time"].astype(str).str.len().value_counts().sort_index()
for L, adet in uzunluk.items():
    ornek = df.loc[df["Start_Time"].astype(str).str.len() == L, "Start_Time"].iloc[0]
    print(f"  {L} karakter: {adet:>9,} satır   örnek: {ornek}")

df["Start_Time"] = pd.to_datetime(df["Start_Time"], format="mixed")

print(f"\nStart_Time -> {df['Start_Time'].dtype}")
print("Tarih aralığı:", df["Start_Time"].min(), "—", df["Start_Time"].max())

 1. parça —  74,548 satır alındı
 2. parça —  70,126 satır alındı
 3. parça —  75,798 satır alındı
 4. parça —  76,230 satır alındı
 5. parça —  75,437 satır alındı
 6. parça —  68,603 satır alındı
 7. parça —  76,381 satır alındı
 8. parça —  94,775 satır alındı
 9. parça — 115,995 satır alındı
10. parça — 115,243 satır alındı
11. parça — 116,719 satır alındı
12. parça — 122,991 satır alındı
13. parça — 123,131 satır alındı
14. parça — 121,793 satır alındı
15. parça —  59,776 satır alındı
16. parça —  32,690 satır alındı

Toplam satır: 1,420,236
Sütun sayısı: 35

State
FL    880192
NY    347960
MN    192084
Name: count, dtype: int64

--- Start_Time metin uzunlukları (dönüşümden önce) ---
  19 karakter: 1,247,864 satır   örnek: 2016-11-30 15:36:03
  26 karakter:    14,480 satır   örnek: 2022-12-14 06:40:00.000000
  29 karakter:   157,892 satır   örnek: 2023-03-31 17:09:16.000000000

Start_Time -> datetime64[ns]
Tarih aralığı: 2016-03-23 02:35:03 — 2023-03-31 23:09:00


## 2. Tekrar eden kayıtların temizlenmesi

Aynı kaza birden çok trafik API'sinden geldiği için veride birden fazla kez
yer alabiliyor: aynı saniye, aynı koordinat, ardışık ID.

**Neden temizlenmeli?** Bu kayıtların bir kopyası eğitim, diğeri test kümesine
düşer; model test aşamasında ezberlediği satırla karşılaşır ve skor gerçekte
olduğundan yüksek çıkar (**veri sızıntısı**).

**Ölçüt:** ID dışındaki tüm sütunlar aynı. Sadece zaman+koordinata bakan daha
gevşek bir ölçüt 4.485 satır daha temizlerdi, ama o satırların Severity
değerleri birbirinden farklı — hangisinin tutulacağı keyfi bir seçim olurdu.
Tüm sütunlar aynıysa böyle bir sorun yok, bilgi kaybı olmuyor.

Temizlik doldurmadan **önce** yapılıyor ki medyanlar tekrarlı dağılımdan
hesaplanmasın. § 3.3'te ikinci bir geçiş var.

In [33]:
# Tekrar tanımı: ID dışındaki TÜM sütunlar aynıysa aynı kazadır.
# ID her satırda benzersiz olduğu için karşılaştırmaya dahil edilmez.
ayirt_edici = [c for c in df.columns if c != "ID"]

tekrarli = df.duplicated(subset=ayirt_edici, keep=False)   # tüm kopyalar
atilacak_tekrar = df.duplicated(subset=ayirt_edici, keep="first")  # ilki hariç

print("=== TEKRAR ANALİZİ ===")
print(f"Tekrara karışan satır : {tekrarli.sum():>9,} ({tekrarli.mean():.1%})")
print(f"Benzersiz olay sayısı : {len(df) - atilacak_tekrar.sum():>9,}")
print(f"Atılacak satır        : {atilacak_tekrar.sum():>9,}")

# --- Kanıt 1: örnek bir tekrar grubu ---
print("\n--- Örnek: aynı kazanın iki kaydı ---")
ornek = df[tekrarli].sort_values(["Start_Time", "Start_Lat"]).head(2)
print(ornek[["ID", "Severity", "Start_Time", "Start_Lat", "Start_Lng",
             "Street", "City", "Temperature(F)"]].to_string(index=False))

# --- Kanıt 2: tekrarlar hangi sınıfta yığılıyor? ---
print("\n--- Severity dağılımı: tekrarlı satırlar vs tüm veri (%) ---")
siniflar = sorted(df["Severity"].unique())
kiyas = pd.DataFrame({
    "tekrarli": df[tekrarli]["Severity"].value_counts(normalize=True) * 100,
    "tum_veri": df["Severity"].value_counts(normalize=True) * 100,
}).reindex(siniflar).fillna(0).round(2)   # o sınıfta hiç tekrar yoksa 0 yaz
kiyas["fark"] = (kiyas["tekrarli"] - kiyas["tum_veri"]).round(2)
print(kiyas.to_string())
print("Pozitif fark = o sınıf tekrarlarda fazla temsil ediliyor.")

# --- Temizle ---
oncesi = len(df)
df = df[~atilacak_tekrar].reset_index(drop=True)

print(f"\n=== SONUÇ ===")
print(f"Satır sayısı: {oncesi:,} -> {len(df):,}  ({len(df)-oncesi:+,})")
print("\nTemizlik sonrası Severity dağılımı (%):")
print((df["Severity"].value_counts(normalize=True).sort_index() * 100).round(2).to_string())

=== TEKRAR ANALİZİ ===
Tekrara karışan satır :   235,843 (16.6%)
Benzersiz olay sayısı : 1,278,270
Atılacak satır        :   141,966

--- Örnek: aynı kazanın iki kaydı ---
       ID  Severity          Start_Time  Start_Lat  Start_Lng     Street  City  Temperature(F)
A-3518246         3 2016-04-20 18:45:25   40.82687  -73.92268 E 161st St Bronx            60.1
A-3518247         3 2016-04-20 18:45:25   40.82687  -73.92268 E 161st St Bronx            60.1

--- Severity dağılımı: tekrarlı satırlar vs tüm veri (%) ---
          tekrarli  tum_veri   fark
Severity                           
1             0.29      0.62  -0.33
2            97.15     83.27  13.88
3             1.86     14.36 -12.50
4             0.69      1.74  -1.05
Pozitif fark = o sınıf tekrarlarda fazla temsil ediliyor.

=== SONUÇ ===
Satır sayısı: 1,420,236 -> 1,278,270  (-141,966)

Temizlik sonrası Severity dağılımı (%):
Severity
1     0.66
2    81.69
3    15.78
4     1.87


## 3. Eksik ve hatalı ölçümler

### 3.1 İmkânsız ölçümler

İstasyon hataları veride gerçek dışı değerler bırakmış: 174 °F, 984 mph,
0.00 inHg. Bunlar **eksik ölçüm** sayılıp `NaN` yapılıyor ve aşağıdaki
doldurmaya bırakılıyor — satır silinmiyor, çünkü hatalı olan tek bir sensör
okuması, kaydın geri kalanı geçerli.

| Sütun | Aralık | | Sütun | Aralık |
|---|---|---|---|---|
| `Temperature(F)` | −40 … 130 | | `Wind_Speed(mph)` | 0 … 100 |
| `Pressure(in)` | 25 … 32 | | `Humidity(%)` | 0 … 100 |
| `Visibility(mi)` | 0 … 20 | | `Precipitation(in)` | 0 … 12 |

Etkilenen kayıt toplamın %0.01'inden az.

### 3.2 Eksik değerlerin doldurulması

Yağışta eksik = "yağış yok" → 0. Diğer sayısallarda medyan (ortalama uç
değerlerden etkilenir). Kategoriklerde "Bilinmiyor".

### 3.3 İkinci tekrar geçişi

Doldurma, § 2'de görünmeyen tekrarları açığa çıkarabilir: aynı kazanın iki
kaydından birinde yağış eksik, diğerinde gerçekten 0.0 ise, doldurma sonrası
satırlar aynı hâle gelir.

Bunlar da aynı kaza: karşılaştırmada `Start_Time`, `Start_Lat`, `Start_Lng`
de var ve bu üç sütun doldurmaya girmiyor.

In [34]:
# --- 0. İMKANSIZ SENSÖR DEĞERLERİNİ EKSİK OLARAK İŞARETLE ---
# Aralık dışındaki ölçüm, ölçümün kendisi hatalı demektir; satırı silmiyoruz,
# sadece o hücreyi NaN yapıp aşağıdaki doldurma adımına bırakıyoruz.
GECERLI_ARALIK = {
    "Temperature(F)":    (-40, 130),
    "Pressure(in)":      ( 25,  32),
    "Visibility(mi)":    (  0,  20),
    "Wind_Speed(mph)":   (  0, 100),
    "Humidity(%)":       (  0, 100),
    "Precipitation(in)": (  0,  12),   # kasırga kaynaklı aşırı yağış için pay bırakıldı
}

print("=== FİZİKSEL OLARAK İMKANSIZ ÖLÇÜMLER ===")
toplam_bozuk = 0
for sutun, (alt, ust) in GECERLI_ARALIK.items():
    bozuk = (df[sutun] < alt) | (df[sutun] > ust)
    n = bozuk.sum()
    toplam_bozuk += n
    if n:
        print(f"{sutun:18s} [{alt:>4}, {ust:>4}] dışında: {n:>5,} kayıt "
              f"| görülen uçlar: {df.loc[bozuk, sutun].min():.1f} … {df.loc[bozuk, sutun].max():.1f}")
    df.loc[bozuk, sutun] = np.nan          # eksik olarak işaretle

print(f"Toplam işaretlenen: {toplam_bozuk:,} hücre "
      f"({toplam_bozuk/len(df)*100:.4f}% — satır silinmedi)\n")

# --- ÖNCE ---
eksik = pd.DataFrame({
    "eksik_sayi": df.isna().sum(),
    "eksik_yuzde": (df.isna().mean() * 100).round(2)
})
eksik = eksik[eksik.eksik_sayi > 0].sort_values("eksik_yuzde", ascending=False)

print("DOLDURMADAN ÖNCE")
print(eksik.to_string())
print()

# --- 1. Yağış: eksik = yağış yoktu ---Boşluk aslında "yağmur yoktu" demek. Gerçekten 0 mm yağış vardı
df["Precipitation(in)"] = df["Precipitation(in)"].fillna(0)

# --- 2. Sayısal: medyan ---. Ortalama yerine onu seçtin çünkü ortalama uç değerlerden etkileniyor
SAYISAL = ["Temperature(F)", "Humidity(%)", "Pressure(in)",
           "Visibility(mi)", "Wind_Speed(mph)"]
for s in SAYISAL:
    df[s] = df[s].fillna(df[s].median())

# --- 3. Kategorik: "Bilinmiyor" ---
METIN = ["Weather_Condition", "Wind_Direction", "City", "Zipcode",
         "Street", "Timezone", "Sunrise_Sunset", "Civil_Twilight",
         "Nautical_Twilight", "Astronomical_Twilight"]
for m in METIN:
    df[m] = df[m].fillna("Bilinmiyor")

# --- SONRA ---
kalan = df.isna().sum()
print("DOLDURMADAN SONRA")
print("Toplam eksik:", kalan.sum())
if kalan.sum() > 0:
    print(kalan[kalan > 0].to_string())

# --- 4. İKİNCİ TEKRAR GEÇİŞİ (bkz. § 3.3) ---
# Doldurma, § 2'de görünmeyen tekrarları açığa çıkarır: aynı kazanın iki
# kaydından birinde ölçüm eksikse (örn. yağış), doldurulunca diğerine
# eşitlenir ve satırlar birebir aynı hale gelir.
#
# Bu satırlar da aynı kazadır: karşılaştırmaya Start_Time, Start_Lat ve
# Start_Lng de dahil ve bu üç sütun doldurma işlemine hiç girmiyor. Yani
# eşleşmenin şartı, saniyesi saniyesine aynı anda aynı koordinatta olmak.
ikinci = df.duplicated(subset=ayirt_edici, keep="first")

print("\n=== DOLDURMA SONRASI İKİNCİ TEKRAR KONTROLÜ ===")
print(f"Açığa çıkan tekrar: {ikinci.sum():,}")

if ikinci.sum():
    oncesi = len(df)
    df = df[~ikinci].reset_index(drop=True)
    print(f"Satır sayısı: {oncesi:,} -> {len(df):,}  ({len(df)-oncesi:+,})")
    print("\nToplam temizlenen tekrar (§2 + §3.3):",
          f"{atilacak_tekrar.sum() + ikinci.sum():,}")
else:
    print("Yeni tekrar çıkmadı.")

=== FİZİKSEL OLARAK İMKANSIZ ÖLÇÜMLER ===
Temperature(F)     [ -40,  130] dışında:    21 kayıt | görülen uçlar: -77.8 … 174.0
Pressure(in)       [  25,   32] dışında:    22 kayıt | görülen uçlar: 0.0 … 58.6
Visibility(mi)     [   0,   20] dışında:     9 kayıt | görülen uçlar: 23.0 … 105.0
Wind_Speed(mph)    [   0,  100] dışında:    25 kayıt | görülen uçlar: 105.0 … 984.0
Toplam işaretlenen: 77 hücre (0.0060% — satır silinmedi)

DOLDURMADAN ÖNCE
                       eksik_sayi  eksik_yuzde
Precipitation(in)          287339        22.48
Wind_Speed(mph)             57793         4.52
Wind_Direction              19713         1.54
Humidity(%)                 18860         1.48
Temperature(F)              17051         1.33
Visibility(mi)              15975         1.25
Weather_Condition           14371         1.12
Pressure(in)                11933         0.93
Sunrise_Sunset               4996         0.39
Civil_Twilight               4996         0.39
Nautical_Twilight            4996 

## 4. İlçe verisinin birleştirilmesi

Nüfus, nüfus yoğunluğu ve kentsel-kırsal sınıfı State + ilçe anahtarı
üzerinden ekleniyor. Sadece ilçe adıyla eşleştirmek hatalı olurdu; ABD'de aynı
adı taşıyan onlarca ilçe var.

**İlçe adı sorunu:** İki kaynak farklı yazıyor — Census "St. Johns", kaza
verisi "Saint Johns"; "DeSoto" ↔ "De Soto". Nokta silip küçük harfe çevirmek
yetmiyor. İki taraf da aynı işlevden geçiriliyor: nokta silinir, `Saint` → `St`
indirgenir, boşluklar kaldırılır (`stjohns`, `desoto`).

Düzeltmeden önce 139 kayıt eşleşmiyordu. Bunları medyanla doldurmak, kaydı
**yanlış bir ilçenin ortalamasıyla** etiketlemek olurdu — sorun eksik veri
değil, anahtar uyuşmazlığıydı.

In [35]:
# Referans tablosunu oku
ref = pd.read_csv(REFERANS)
print("Referans tablosu:", ref.shape)
print(ref.columns.tolist())
print()

def ilce_anahtari(s):
    """İki kaynağın ilçe adını ortak biçime indirger.
    'St. Johns County' -> 'stjohns'   |   'Saint Johns' -> 'stjohns'
    'DeSoto'           -> 'desoto'    |   'De Soto'     -> 'desoto'
    """
    s = str(s).lower()
    s = s.replace(".", "")                 # St. -> St
    s = re.sub(r"^saint\s+", "st ", s)     # Saint Johns -> st johns
    s = re.sub(r"\s+", "", s)              # st johns -> stjohns
    return s

# Aynı işlev HER İKİ tarafa da uygulanır — eşleşmenin şartı budur
df["county_key"]  = df["County"].apply(ilce_anahtari)
ref["county_key"] = ref["county_key"].apply(ilce_anahtari)

# Anahtar gerçekten benzersiz mi? (aynı eyalette iki ilçe aynı anahtara düşerse
# birleştirme satır çoğaltır — sessizce olmasın diye önden kontrol ediyoruz)
cakisma = ref.duplicated(["State", "county_key"]).sum()
print("Referansta çakışan anahtar:", cakisma)
assert cakisma == 0, "Anahtar benzersiz değil, birleştirme satır çoğaltır!"

# Birleştir
oncesi = len(df)
df = df.merge(
    ref[["State", "county_key", "nufus_2022",
         "nufus_yogunlugu", "kentsel_kirsal"]],
    on=["State", "county_key"],#Sadece ilçe adıyla eşleştirseydik felaket olurdu. ABD'de 31 tane "Washington County" var. Farklı eyaletlerde, farklı yerler.
    how="left"
)

eslesmeyen = df["nufus_2022"].isna().sum()
print("Satır sayısı:", f"{oncesi:,}", "→", f"{len(df):,}")
print("Eşleşmeyen kayıt:", eslesmeyen, f"({eslesmeyen/len(df):.2%})")

if eslesmeyen:
    print("\n⚠ Eşleşmeyen ilçeler:")
    print(df[df.nufus_2022.isna()].groupby(["State", "County"]).size().to_string())
else:
    print("✓ Tüm kayıtlar ilçe referansıyla eşleşti.")

assert len(df) == oncesi, "Birleştirme satır sayısını değiştirdi!"
print()
print(df[["State", "County", "nufus_2022",
          "nufus_yogunlugu", "kentsel_kirsal"]].head(5).to_string())

Referans tablosu: (3138, 10)
['State', 'county_key', 'GEOID', 'NAME', 'STNAME', 'nufus_2022', 'alan_sqmi', 'nufus_yogunlugu', 'kentsel_kirsal', 'birlesik_kayit']

Referansta çakışan anahtar: 0
Satır sayısı: 1,278,270 → 1,278,270
Eşleşmeyen kayıt: 0 (0.00%)
✓ Tüm kayıtlar ilçe referansıyla eşleşti.

  State        County  nufus_2022  nufus_yogunlugu  kentsel_kirsal
0    FL  Hillsborough     1523839          1490.25             1.0
1    FL  Hillsborough     1523839          1490.25             1.0
2    FL    Miami-Dade     2713415          1428.15             1.0
3    FL    Miami-Dade     2713415          1428.15             1.0
4    FL       Broward     1966237          1634.58             2.0


## 5. Öznitelik mühendisliği

`Start_Time`'dan zaman bileşenleri, `Street`'ten yol tipi, `Weather_Condition`
'dan gruplandırılmış hava durumu türetiliyor.

Üçünde de kaynak sütun **silinmiyor**, yenisi yanına ekleniyor: 02 ham
ayrıntıya erişebilsin, 03 sadeleştirilmiş biçimi kullansın.

### 5.1 Zaman değişkenleri

Ham zaman damgası modelde kullanılamaz — her kaza farklı bir ana denk geldiği
için sütun neredeyse benzersiz, tekrar eden örüntü yok. Bileşenleri ise
tekrar eder: "saat 17", "cumartesi", "kış".

In [36]:
# Start_Time zaten § 1.1'de tarih tipine çevrildi; burada sadece parçalanıyor.
df["saat"]       = df["Start_Time"].dt.hour
df["gun"]        = df["Start_Time"].dt.dayofweek     # 0=Pazartesi
df["ay"]         = df["Start_Time"].dt.month
df["yil"]        = df["Start_Time"].dt.year
df["hafta_sonu"] = df["gun"] >= 5
df["yogun_saat"] = df["saat"].isin([7, 8, 9, 16, 17, 18])

def mevsim(ay):
    if ay in [12, 1, 2]:  return "Kis"
    if ay in [3, 4, 5]:   return "Ilkbahar"
    if ay in [6, 7, 8]:   return "Yaz"
    return "Sonbahar"

df["mevsim"] = df["ay"].apply(mevsim)

print("Boyut:", df.shape)
print()
print("Mevsim dağılımı:")
print(df["mevsim"].value_counts())
print()
print("Örnek dönüşüm:")
print(df[["Start_Time", "saat", "gun", "hafta_sonu",
          "yogun_saat", "mevsim"]].head(3).to_string())

Boyut: (1278270, 46)

Mevsim dağılımı:
mevsim
Kis         387882
Sonbahar    345207
Yaz         277000
Ilkbahar    268181
Name: count, dtype: int64

Örnek dönüşüm:
           Start_Time  saat  gun  hafta_sonu  yogun_saat    mevsim
0 2016-11-30 15:36:03    15    2       False       False  Sonbahar
1 2016-11-30 16:25:35    16    2       False        True  Sonbahar
2 2016-11-30 16:40:31    16    2       False        True  Sonbahar


### 5.2 Yol tipi — ilk deneme

Sokak isimlerinden düzenli ifadelerle yol tipi çıkarılıyor. İlk kural setinde
sadece numaralı yollar (I-75, US-35) ve yaygın şehir içi ekleri (Ave, Blvd,
St) hedeflendi.

In [37]:
def yol_tipi_v1(s):
    s = str(s)
    if re.search(r"\bI-\d+", s):                              return "Otoyol"
    if re.search(r"\bUS-\d+|US Highway", s):                  return "Federal yol"
    if re.search(r"State (Route|Rte)|\b[A-Z]{2}-\d+", s):     return "Eyalet yolu"
    if re.search(r"County (Hwy|Road)|\bCR-\d+", s):           return "Ilce yolu"
    if re.search(r"\b(Dr|Ave|St|Ln|Ct|Blvd|Rd|Way|Pl)\b", s): return "Sehir ici"
    return "Diger"

v1 = df["Street"].apply(yol_tipi_v1)

print("İLK DENEME")
print(v1.value_counts())
print()
print("Sınıflandırılamayan oran:", f"{(v1 == 'Diger').mean():.1%}")
print()
print("En sık sınıflandırılamayan sokaklar:")
print(df.loc[v1 == "Diger", "Street"].value_counts().head(20).to_string())

İLK DENEME
Street
Sehir ici      560371
Diger          350104
Otoyol         278673
Federal yol     42031
Eyalet yolu     41432
Ilce yolu        5659
Name: count, dtype: int64

Sınıflandırılamayan oran: 27.4%

En sık sınıflandırılamayan sokaklar:
Street
Florida's Tpke S         9090
Brooklyn Queens Expy     8711
Florida's Tpke N         8524
Palmetto Expy S          7109
Long Island Expy W       7086
Long Island Expy E       6717
Palmetto Expy N          5600
 S Dixie Hwy             4510
 S Orange Blossom Trl    4420
Florida's Tpke           4088
Southern State Pkwy E    3655
Ronald Reagan Tpke       3273
Southern State Pkwy W    3214
W Beltway S              3148
Major Deegan Expy N      3087
Dolphin Expy E           2960
E Beltway N              2756
Adirondack Northway N    2473
W Beltway N              2433
E Beltway S              2311


### 5.3 Yol tipi — genişletilmiş kural seti

Sınıflandırılamayanlara bakınca çoğunun Turnpike, Expressway, Parkway,
Thruway gibi **isimli** hızlı yollar olduğu görüldü — numaralı kalıp
içermedikleri için ilk kural setine takılmamışlar. Kurallar genişletildi.

In [38]:
def yol_tipi(s):
    s = str(s)
    # Numaralı eyaletler arası otoyol
    if re.search(r"\bI-\d+", s):
        return "Otoyol"
    # İsimli hızlı yollar — ilk denemede kaçanlar
    if re.search(r"\b(Expy|Expressway|Tpke|Turnpike|Pkwy|Parkway|Fwy|Freeway|Thruway|Beltway|Northway|Skyway|Causeway)\b", s, re.I):#re.I eklendi. "Ignore case" — büyük/küçük harf farkını yok sayar.
        return "Otoyol"
    if re.search(r"\bUS-\d+|US Highway", s):
        return "Federal yol"
    if re.search(r"State (Route|Rte)|\b[A-Z]{2}-\d+", s):
        return "Eyalet yolu"
    if re.search(r"County (Hwy|Road)|\bCR-\d+", s):
        return "Ilce yolu"
    if re.search(r"\b(Hwy|Highway)\b", s, re.I):
        return "Federal yol"
    if re.search(r"\b(Dr|Drive|Ave|Avenue|St|Street|Ln|Lane|Ct|Court|Blvd|Boulevard|Rd|Road|Way|Pl|Place|Pike|Trl|Trail|Cir|Circle|Ter|Terrace|Loop|Row|Aly|Alley|Sq|Square|Plz|Plaza|Bridge|Tunnel)\b", s, re.I):
        return "Sehir ici"
    return "Diger"

df["yol_tipi"] = df["Street"].apply(yol_tipi)

print("GENİŞLETİLMİŞ KURAL SETİ")
print(df["yol_tipi"].value_counts())
print()
print("Sınıflandırılamayan oran:", f"{(df['yol_tipi']=='Diger').mean():.1%}")
print("(ilk denemede %27.2 idi)")
print()
print("Hâlâ sınıflandırılamayanlar:")
print(df.loc[df["yol_tipi"]=="Diger", "Street"].value_counts().head(10).to_string())

GENİŞLETİLMİŞ KURAL SETİ
yol_tipi
Sehir ici      605633
Otoyol         480646
Federal yol    105266
Eyalet yolu     41432
Diger           39634
Ilce yolu        5659
Name: count, dtype: int64

Sınıflandırılamayan oran: 3.1%
(ilk denemede %27.2 idi)

Hâlâ sınıflandırılamayanlar:
Street
New England Trwy S             2225
New York Trwy W                1737
Bilinmiyor                     1677
George Washington Brg          1594
Central Florida Greeneway S    1310
New York Trwy N                1270
Central Florida Greeneway N     742
Shakopee Byp N                  742
New England Trwy N              714
Route 9                         688


### 5.4 Hava durumu gruplaması

`Weather_Condition` 101 farklı değer içeriyor ve çoğu aynı olayın farklı
ayrıntı düzeyi: "Light Rain", "Heavy Rain", "Rain Showers" hepsi yağış.

**Neden gruplanıyor?** Kategorilerin çoğu birkaç yüz kayıtta görülüyor; bu
kadar seyrek bir kategoride hesaplanan oran rastlantısal dalgalanmadan ibaret,
model onu örüntü sanar. Ayrıca araştırma sorusu "yağmur kazaları ağırlaştırıyor
mu?" düzeyinde, "Light Rain ile Light Drizzle farklı mı?" düzeyinde değil.

Kural sırası önemli: liste yukarıdan aşağı işletildiği için birden fazla
kurala uyan kayıt daha kritik olan koşula atanır.

Ham sütun korunuyor; 02 en sık 12 ham kategoriyi ayrı inceliyor.

In [39]:
def hava_grupla(s):
    s = str(s).lower()
    # Sıra kritik: üstteki kural önce yakalar.
    # "Light Snow with Thunder" -> Firtina değil Kar_Buz olmamalı; önce sis/kar/fırtına
    # gibi görüş ve zemin riski yüksek koşullar sınıflandırılır.
    if "fog" in s or "mist" in s or "haze" in s:                      return "Sis"
    if "snow" in s or "sleet" in s or "ice" in s or "freezing" in s:  return "Kar_Buz"
    if "thunder" in s or "t-storm" in s or "storm" in s:              return "Firtina"
    if "rain" in s or "drizzle" in s or "shower" in s:                return "Yagmur"
    if "cloud" in s or "overcast" in s:                               return "Bulutlu"
    if "clear" in s or "fair" in s:                                   return "Acik"
    if "wind" in s or "dust" in s or "sand" in s or "smoke" in s:     return "Ruzgar_Toz"
    return "Diger"

df["hava_durumu"] = df["Weather_Condition"].apply(hava_grupla)

print("HAVA DURUMU GRUPLAMASI")
print(f"Ham kategori sayısı : {df['Weather_Condition'].nunique()}")
print(f"Grup sayısı         : {df['hava_durumu'].nunique()}")
print()

ozet = pd.DataFrame({"kayit": df["hava_durumu"].value_counts()})
ozet["yuzde"] = (ozet["kayit"] / len(df) * 100).round(2)
# Her grubun ciddi kaza oranı — gruplamanın anlamlı olup olmadığını gösterir
ozet["ciddi_%"] = (df.assign(ciddi=df["Severity"] >= 3)
                     .groupby("hava_durumu")["ciddi"].mean() * 100).round(2)
print(ozet.to_string())

print("\nHangi ham kategoriler hangi gruba düştü? (grup başına en sık 3 tanesi)")
for g in ozet.index:
    ilk3 = df.loc[df.hava_durumu == g, "Weather_Condition"].value_counts().head(3)
    print(f"  {g:11s} <- " + ", ".join(f"{k} ({v:,})" for k, v in ilk3.items()))

print("\nSütun sayısı:", df.shape[1])

HAVA DURUMU GRUPLAMASI
Ham kategori sayısı : 101
Grup sayısı         : 8

              kayit  yuzde  ciddi_%
hava_durumu                        
Bulutlu      589081  46.08    20.45
Acik         499483  39.07    14.05
Yagmur        78847   6.17    22.16
Kar_Buz       42739   3.34    16.41
Firtina       30990   2.42    15.72
Sis           18951   1.48    14.15
Diger         17782   1.39    15.88
Ruzgar_Toz      397   0.03    19.40

Hangi ham kategoriler hangi gruba düştü? (grup başına en sık 3 tanesi)
  Bulutlu     <- Mostly Cloudy (216,869), Partly Cloudy (150,916), Cloudy (123,390)
  Acik        <- Fair (399,732), Clear (94,779), Fair / Windy (4,972)
  Yagmur      <- Light Rain (55,499), Rain (10,894), Heavy Rain (5,730)
  Kar_Buz     <- Light Snow (33,541), Snow (3,672), Light Snow / Windy (1,960)
  Firtina     <- Thunder in the Vicinity (7,233), Thunder (6,181), T-Storm (4,989)
  Sis         <- Fog (13,084), Haze (3,569), Patches of Fog (689)
  Diger       <- Bilinmiyor (14,371), Wi

## 6. Kalite denetimi ve kaydetme

Kaydetmeden önce altı denetim `assert` ile yapılıyor: eksik değer, tekrar eden
kayıt, eşleşmeyen ilçe, sensör aralıkları, türetilmiş sütunlar, hedef değişken.
Biri sağlanmazsa notebook durur — bozuk dosya sessizce kaydedilmez.

`county_key` sadece birleştirme anahtarıydı, çıktıya girmiyor.

> Bu dosya modellemeye hazırdır; 02 ve 03 ek temizlik yapmaz.

In [40]:
CIKTI = PROC + "temiz_veri.parquet"

# Birleştirme anahtarı görevini tamamladı, çıktıya girmiyor
df = df.drop(columns=["county_key"])

# =========================  KALİTE KONTROLÜ  =========================
print("=== KAYIT ÖNCESİ DENETİM ===")
hatalar = []

# 1) Eksik değer kalmamalı
n_eksik = df.isna().sum().sum()
print(f"{'Eksik değer':28s}: {n_eksik:>10,}")
if n_eksik: hatalar.append(f"{n_eksik} eksik değer var")

# 2) Tekrar eden kayıt kalmamalı
n_tekrar = df.duplicated(subset=[c for c in df.columns if c != "ID"]).sum()
print(f"{'Tekrar eden kayıt':28s}: {n_tekrar:>10,}")
if n_tekrar: hatalar.append(f"{n_tekrar} tekrar eden kayıt var")

# 3) İlçe eşleşmesi tam olmalı
n_ilce = df["nufus_2022"].isna().sum()
print(f"{'Eşleşmeyen ilçe':28s}: {n_ilce:>10,}")
if n_ilce: hatalar.append(f"{n_ilce} kayıt ilçe referansıyla eşleşmedi")

# 4) Sensör değerleri aralıkta olmalı
for sutun, (alt, ust) in GECERLI_ARALIK.items():
    n = ((df[sutun] < alt) | (df[sutun] > ust)).sum()
    if n: hatalar.append(f"{sutun}: {n} kayıt [{alt},{ust}] dışında")
print(f"{'Sensör aralık ihlali':28s}: {sum(1 for h in hatalar if ':' in h):>10,}")

# 5) Türetilmiş sütunlar yerinde olmalı
BEKLENEN = ["saat", "gun", "ay", "yil", "hafta_sonu", "yogun_saat",
            "mevsim", "yol_tipi", "hava_durumu",
            "nufus_2022", "nufus_yogunlugu", "kentsel_kirsal"]
eksik_sutun = [c for c in BEKLENEN if c not in df.columns]
print(f"{'Eksik türetilmiş sütun':28s}: {len(eksik_sutun):>10,}")
if eksik_sutun: hatalar.append(f"türetilmiş sütun eksik: {eksik_sutun}")

# 6) Hedef değişken geçerli olmalı
gecersiz_y = (~df["Severity"].isin([1, 2, 3, 4])).sum()
print(f"{'Geçersiz Severity':28s}: {gecersiz_y:>10,}")
if gecersiz_y: hatalar.append(f"{gecersiz_y} kayıtta Severity 1-4 dışında")

assert not hatalar, "DENETİM BAŞARISIZ:\n - " + "\n - ".join(hatalar)
print("\n✓ Tüm denetimler geçildi — veri modellemeye hazır.\n")

# =========================  KAYDET  =========================
df.to_parquet(CIKTI, index=False)

boyut = os.path.getsize(CIKTI) / 1024**2
print("Kaydedildi:", CIKTI)
print(f"Dosya boyutu: {boyut:.1f} MB")
print(f"Satır: {len(df):,}   Sütun: {df.shape[1]}")
print()
print("Severity dağılımı (%):")
print((df["Severity"].value_counts(normalize=True).sort_index() * 100).round(2).to_string())
print()
print("Sütunlar:")
print(df.columns.tolist())

=== KAYIT ÖNCESİ DENETİM ===
Eksik değer                 :          0
Tekrar eden kayıt           :          0
Eşleşmeyen ilçe             :          0
Sensör aralık ihlali        :          0
Eksik türetilmiş sütun      :          0
Geçersiz Severity           :          0

✓ Tüm denetimler geçildi — veri modellemeye hazır.

Kaydedildi: ../data/processed/temiz_veri.parquet
Dosya boyutu: 56.2 MB
Satır: 1,278,270   Sütun: 47

Severity dağılımı (%):
Severity
1     0.66
2    81.69
3    15.78
4     1.87

Sütunlar:
['ID', 'Severity', 'Start_Time', 'Start_Lat', 'Start_Lng', 'Street', 'City', 'County', 'State', 'Zipcode', 'Timezone', 'Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_T